In [1]:
import pandas as pd 
import numpy as np
import holidays

In [2]:
def generate_exogenous_calendar(start_year=2016, end_year=2026):
    # 1. Ambil database hari libur nasional Indonesia (2013 - 2026)
    id_holidays = holidays.Indonesia(years=range(start_year, end_year + 1))
    
    # Buat rentang tanggal harian dari 1 Jan n2013 s.d. 31 Des 2026
    daily_dates = pd.date_range(start=f"{start_year}-01-01", end=f"{end_year}-12-31", freq='D')
    df_daily = pd.DataFrame({'date': daily_dates})
    
    # Tandai hari Senin-Jumat (dayofweek < 5) dan bukan hari libur nasional
    df_daily['is_weekday'] = df_daily['date'].dt.dayofweek < 5
    df_daily['is_holiday'] = df_daily['date'].dt.date.isin(id_holidays)
    df_daily['is_workday'] = df_daily['is_weekday'] & (~df_daily['is_holiday'])
    
    # Agregasi ke tingkat Bulanan (Monthly Start / MS)
    df_daily['year_month'] = df_daily['date'].dt.to_period('M')
    df_monthly = df_daily.groupby('year_month').agg(
        total_hari=('date', 'count'),
        jumlah_hari_kerja=('is_workday', 'sum'),
        jumlah_libur_nasional_weekday=('is_holiday', lambda x: (x & df_daily.loc[x.index, 'is_weekday']).sum())
    ).reset_index()
    
    df_monthly['tanggal'] = df_monthly['year_month'].dt.to_timestamp()
    
    # 2. Penanda Lebaran (Bulan jatuhnya Idul Fitri 2013 - 2026)
    lebaran_months = [
        '2013-08', '2014-07', '2015-07', '2016-07', '2017-06',
        '2018-06', '2019-06', '2020-05', '2021-05', '2022-05',
        '2023-04', '2024-04', '2025-03', '2026-03'
    ]
    df_monthly['is_lebaran'] = df_monthly['year_month'].astype(str).isin(lebaran_months).astype(int)
    
    # 3. Penanda Periode COVID-19 (2020 - 2022)
    # Opsi A: Full tahun 2020-2022 sesuai tulisan di dokumen (2020-2022)
    df_monthly['is_covid_2020_2022'] = df_monthly['tanggal'].dt.year.isin([2020, 2021, 2022]).astype(int)
    
    # Opsi B (Lebih presisi untuk Time Series): Maret 2020 (kasus pertama/PSBB) s.d. Desember 2022 (PPKM dicabut)
    df_monthly['is_covid_mar20_dec22'] = (
        (df_monthly['tanggal'] >= '2020-03-01') & 
        (df_monthly['tanggal'] <= '2022-12-31')
    ).astype(int)
    
    # Rapikan urutan kolom
    cols = [
        'year_month', 'tanggal', 'total_hari', 'jumlah_hari_kerja', 
        'jumlah_libur_nasional_weekday', 'is_lebaran', 
        'is_covid_2020_2022', 'is_covid_mar20_dec22'
    ]
    return df_monthly[cols]

# Jalankan dan simpan ke CSV
df_kalender = generate_exogenous_calendar(2016, 2026)
df_kalender.to_csv("kalender_eksogen_krl_2013_2026.csv", index=False)
print(df_kalender.head(12))

   year_month    tanggal  total_hari  jumlah_hari_kerja  \
0     2016-01 2016-01-01          31                 20   
1     2016-02 2016-02-01          29                 20   
2     2016-03 2016-03-01          31                 21   
3     2016-04 2016-04-01          30                 21   
4     2016-05 2016-05-01          31                 20   
5     2016-06 2016-06-01          30                 21   
6     2016-07 2016-07-01          31                 19   
7     2016-08 2016-08-01          31                 22   
8     2016-09 2016-09-01          30                 21   
9     2016-10 2016-10-01          31                 21   
10    2016-11 2016-11-01          30                 22   
11    2016-12 2016-12-01          31                 21   

    jumlah_libur_nasional_weekday  is_lebaran  is_covid_2020_2022  \
0                               1           0                   0   
1                               1           0                   0   
2                        